# 🌿 CLIP Plant Disease Classifier — Training Notebook

This notebook fine-tunes **OpenAI CLIP (ViT-B/16)** on the PlantVillage dataset to classify plant diseases from leaf images.

**What we do:**
1. Load and configure CLIP
2. Partially unfreeze the model (last transformer block only)
3. Map class labels → natural language prompts
4. Train with Cross-Entropy loss + AdamW optimizer
5. Save the best model and visualize results

---
**Dataset:** [PlantVillage](https://github.com/spMohanty/PlantVillage-Dataset) — 38 classes of healthy and diseased plant leaves  
**Model:** CLIP ViT-B/16 (~86M parameters, ~5.2M trainable after partial unfreeze)

## Section 1 — Imports

We import standard PyTorch libraries, PIL for image loading, CLIP for the model, and visualization libraries.

- `clip` — OpenAI's CLIP model (install via `pip install git+https://github.com/openai/CLIP.git`)
- `tqdm` — progress bars for training loops
- `sklearn` — for classification report and confusion matrix
- `seaborn` — for heatmap visualizations

In [2]:
import torch
import clip
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os, json, random
from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
from datetime import datetime

print('All imports successful!')
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

All imports successful!
PyTorch version: 2.7.1+cu118
CUDA available: True


## Section 2 — Configuration

All hyperparameters and paths are defined here in one place for easy tuning.

| Parameter | Value | Reason |
|-----------|-------|--------|
| `EPOCHS` | 10 | Enough for convergence without overfitting |
| `BATCH_SIZE` | 32 | Good GPU utilization; double from baseline |
| `LR` | 1e-6 | Very small — we don't want to destroy CLIP's pre-training |
| `WEIGHT_DECAY` | 0.05 | AdamW regularization to prevent overfitting |
| `GRAD_CLIP` | 1.0 | Prevents gradient explosion during training |

> **Why such a small learning rate?** CLIP is already pre-trained on 400M image-text pairs. A large LR would overwrite valuable features. We only nudge the last block.

In [ ]:
# ── Path to your PlantVillage 'color' folder ──
DATASET_DIR  = r"C:\Users\abhis\Desktop\plantvillage dataset\color"

EPOCHS       = 10
BATCH_SIZE   = 32
LR           = 1e-6
WEIGHT_DECAY = 0.05
GRAD_CLIP    = 1.0
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"

# Output folder for saved plots
SAVE_DIR = "training_outputs"
os.makedirs(SAVE_DIR, exist_ok=True)

print(f'Device : {DEVICE}')
print(f'Epochs : {EPOCHS} | Batch: {BATCH_SIZE} | LR: {LR}')
print(f'Output : {SAVE_DIR}/')

## Section 3 — Label-to-Prompt Conversion

CLIP works with **text prompts**, not integer class indices. We convert raw folder names like:

```
Tomato___Late_blight  →  "a Tomato leaf with Late blight disease"
Apple___healthy       →  "a healthy Apple leaf with no disease"
```

This is the core idea of CLIP for classification: instead of a learned linear head, we use **natural language** to describe each class. The model computes similarity between the image embedding and all prompt embeddings.

In [ ]:
def label_to_prompt(label):
    """Convert a folder name like 'Tomato___Late_blight' into a natural language prompt."""
    parts     = label.split("___")
    plant     = parts[0].replace("_", " ")
    condition = parts[1].replace("_", " ") if len(parts) > 1 else "healthy"
    if "healthy" in condition.lower():
        return f"a healthy {plant} leaf with no disease"
    return f"a {plant} leaf with {condition} disease"

# Quick test
test_labels = ["Tomato___Late_blight", "Apple___healthy", "Corn_(maize)___Common_rust_"]
for lbl in test_labels:
    print(f'  {lbl:40s} →  "{label_to_prompt(lbl)}"')

## Section 4 — Dataset Class

The `PlantDataset` class handles:

- **Discovery**: walks the dataset folder and collects all image paths + labels
- **80/20 train/val split**: reproducible with `random.seed(42)`
- **CLIP preprocessing**: uses CLIP's built-in `preprocess` (resize to 224, normalize, etc.)
- **Augmentation** (train only): random horizontal flip to improve generalization
- **Error handling**: broken images return a zero tensor instead of crashing

**Important:** The split is done at the file level (not folder level), so both train and val contain samples from every class.

In [ ]:
class PlantDataset(Dataset):
    """
    Loads images from a PlantVillage-style folder structure:
        root_dir/
            Apple___Apple_scab/  ← folder name = class label
                image1.jpg
                image2.jpg
            Apple___healthy/
                ...
    """
    def __init__(self, root_dir, preprocess, split="train", val_ratio=0.2, augment=False):
        self.preprocess   = preprocess
        self.split        = split
        # Only augment during training
        self.augment      = augment and (split == "train")

        # Build sorted class list (ensures consistent ordering every run)
        self.classes      = sorted([d for d in os.listdir(root_dir)
                                    if os.path.isdir(os.path.join(root_dir, d))])
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        self.idx_to_class = {i: c for c, i in self.class_to_idx.items()}

        # Collect all image paths
        all_samples = []
        for cls in self.classes:
            cls_dir = os.path.join(root_dir, cls)
            for fname in os.listdir(cls_dir):
                if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                    all_samples.append((os.path.join(cls_dir, fname), cls))

        # Reproducible 80/20 split
        random.seed(42)
        random.shuffle(all_samples)
        split_idx    = int(len(all_samples) * (1 - val_ratio))
        self.samples = all_samples[:split_idx] if split == "train" else all_samples[split_idx:]

        # Simple augmentation: random horizontal flip
        self.flip = lambda img: img.transpose(Image.FLIP_LEFT_RIGHT) if random.random() > 0.5 else img

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        try:
            img = Image.open(path).convert("RGB")
            if self.augment:
                img = self.flip(img)
            image = self.preprocess(img)
        except Exception as e:
            print(f"Warning: Could not load {path}: {e}")
            image = torch.zeros(3, 224, 224)  # Return blank image on error
        return image, self.class_to_idx[label]

## Section 5 — Visualization Functions

Three plotting functions are defined:

1. **`plot_training_curves`** — 4-panel figure with accuracy/loss line plots + bar chart + summary
2. **`plot_confusion_matrix`** — raw counts + normalized (row %) heatmaps side by side; also prints a full classification report
3. **`plot_per_class_accuracy`** — horizontal bar chart coloured by threshold (green > 80%, orange > 60%, red otherwise)

All figures are saved to `training_outputs/` as PNG files.

In [ ]:
def plot_training_curves(history, save_path):
    """4-panel training metrics figure: accuracy, loss, bar comparison, summary."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('CLIP Plant Disease Classification - Training Metrics', fontsize=14, fontweight='bold')
    epochs = range(1, len(history['train_acc']) + 1)

    # Panel 1: Accuracy lines
    ax = axes[0, 0]
    ax.plot(epochs, history['train_acc'], 'b-o', label='Train Accuracy', linewidth=2, markersize=6)
    ax.plot(epochs, history['val_acc'],   'r-s', label='Val Accuracy',   linewidth=2, markersize=6)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy (%)'); ax.set_title('Accuracy Over Epochs')
    ax.legend(); ax.grid(True, alpha=0.3); ax.set_ylim([0, 105])

    # Panel 2: Loss lines
    ax = axes[0, 1]
    ax.plot(epochs, history['train_loss'], 'b-o', label='Train Loss', linewidth=2, markersize=6)
    ax.plot(epochs, history['val_loss'],   'r-s', label='Val Loss',   linewidth=2, markersize=6)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss'); ax.set_title('Loss Over Epochs')
    ax.legend(); ax.grid(True, alpha=0.3)

    # Panel 3: Bar comparison
    ax = axes[1, 0]
    x = np.arange(len(epochs)); width = 0.35
    ax.bar(x - width/2, history['train_acc'], width, label='Train', color='steelblue', alpha=0.8)
    ax.bar(x + width/2, history['val_acc'],   width, label='Val',   color='coral',     alpha=0.8)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy (%)'); ax.set_title('Train vs Val Accuracy (Bar)')
    ax.set_xticks(x); ax.set_xticklabels(epochs); ax.legend(); ax.grid(True, alpha=0.3, axis='y')

    # Panel 4: Summary text box
    ax = axes[1, 1]; ax.axis('off')
    best_epoch = np.argmax(history['val_acc']) + 1
    best_acc   = max(history['val_acc'])
    info_text  = f"""TRAINING SUMMARY

Best Val Accuracy : {best_acc:.2f}%
Best Epoch        : {best_epoch}/{len(epochs)}
Final Train Acc   : {history['train_acc'][-1]:.2f}%
Final Val Acc     : {history['val_acc'][-1]:.2f}%

Trainable Params  : ~5.2M / 86M
Learning Rate     : {LR}
Batch Size        : {BATCH_SIZE}"""
    ax.text(0.5, 0.5, info_text, transform=ax.transAxes, fontsize=11,
            verticalalignment='center', horizontalalignment='center',
            bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor='orange', linewidth=2))

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='white')
    print(f'\n✓ Training curves saved to: {save_path}')
    plt.show()


def plot_confusion_matrix(y_true, y_pred, class_names, save_path):
    """Side-by-side confusion matrices: raw counts + row-normalised percentages."""
    cm      = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    cm_norm = np.nan_to_num(cm_norm)

    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    fig.suptitle('Confusion Matrix Analysis', fontsize=14, fontweight='bold')

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
                xticklabels=class_names, yticklabels=class_names, annot_kws={"size": 8})
    axes[0].set_title('Confusion Matrix (Raw Counts)')
    axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

    sns.heatmap(cm_norm, annot=True, fmt='.1%', cmap='RdYlGn', ax=axes[1],
                xticklabels=class_names, yticklabels=class_names,
                vmin=0, vmax=1, annot_kws={"size": 8})
    axes[1].set_title('Confusion Matrix (Normalized by Row)')
    axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')

    for ax in axes:
        plt.setp(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
        plt.setp(ax.get_yticklabels(), rotation=0, fontsize=8)

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='white')
    print(f'✓ Confusion matrix saved to: {save_path}')
    plt.show()

    print('\n' + '='*60)
    print('CLASSIFICATION REPORT')
    print('='*60)
    print(classification_report(y_true, y_pred, target_names=class_names, digits=3))
    return cm, cm_norm


def plot_per_class_accuracy(cm_norm, class_names, save_path):
    """Horizontal bar chart of per-class accuracy, colour-coded green/orange/red."""
    per_class_acc = np.diag(cm_norm) * 100
    colors = ['#4CAF50' if a > 80 else '#FF9800' if a > 60 else '#F44336' for a in per_class_acc]

    fig, ax = plt.subplots(figsize=(14, max(6, len(class_names) * 0.4)))
    bars = ax.barh(range(len(class_names)), per_class_acc, color=colors, edgecolor='black', alpha=0.8)
    ax.set_yticks(range(len(class_names)))
    ax.set_yticklabels(class_names, fontsize=9)
    ax.set_xlabel('Accuracy (%)', fontsize=11)
    ax.set_title('Per-Class Accuracy on Validation Set', fontsize=13, fontweight='bold', pad=15)
    ax.set_xlim([0, 105]); ax.grid(True, alpha=0.3, axis='x')

    for bar, acc in zip(bars, per_class_acc):
        ax.text(acc + 1, bar.get_y() + bar.get_height()/2, f'{acc:.1f}%', va='center', fontsize=9)

    avg_acc = np.mean(per_class_acc)
    ax.axvline(avg_acc, color='blue', linestyle='--', linewidth=2, label=f'Average: {avg_acc:.1f}%')
    ax.legend(loc='lower right')

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='white')
    print(f'✓ Per-class accuracy saved to: {save_path}')
    plt.show()

## Section 6 — Load CLIP Model

We load `ViT-B/16` (Vision Transformer, patch size 16). This model has ~86M parameters.

**Critical step:** Convert to `float32`. By default, CLIP loads in `float16` on GPU, which can cause NaN losses during fine-tuning. Converting to float32 makes training stable.

**Partial unfreezing strategy:**
- Freeze everything (all 86M params)
- Then unfreeze only the **last visual transformer block** (`resblocks.11`) + `ln_post` + `visual.proj`
- Result: ~5.2M trainable parameters (6% of total)

This prevents catastrophic forgetting of CLIP's powerful visual representations.

In [ ]:
print('Loading CLIP ViT-B/16...')
model, preprocess = clip.load("ViT-B/16", device=DEVICE)

# CRITICAL: Convert from float16 to float32 to avoid NaN losses
model = model.float()
print('Model converted to float32 ✓')

# Step 1: Freeze ALL parameters
for param in model.parameters():
    param.requires_grad = False

# Step 2: Selectively unfreeze last visual block + projection head
UNFREEZE_LAYERS = [
    "visual.transformer.resblocks.11",  # Last transformer block
    "visual.ln_post",                   # Layer norm after visual encoder
    "visual.proj",                      # Linear projection to embedding space
]
for name, param in model.named_parameters():
    if any(x in name for x in UNFREEZE_LAYERS):
        param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable:,} / {total:,} ({trainable/total*100:.1f}%)')

## Section 7 — Create Datasets & DataLoaders

We instantiate train and validation datasets:
- Train: 80% of data, with random horizontal flip augmentation
- Val: 20% of data, no augmentation (deterministic evaluation)

Class labels are saved to `clip_labels.json` so the inference app can load them later without needing the dataset directory.

> **Note:** `num_workers=0` avoids multiprocessing issues on Windows. On Linux you can set it to 4+ for faster loading.

In [ ]:
print('Loading dataset...')
train_ds = PlantDataset(DATASET_DIR, preprocess, split="train", augment=True)
val_ds   = PlantDataset(DATASET_DIR, preprocess, split="val",   augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)

num_classes = len(train_ds.classes)
print(f'Train: {len(train_ds):,} images | Val: {len(val_ds):,} images | Classes: {num_classes}')

# Save class list + prompts for the inference app
with open("clip_labels.json", "w") as f:
    json.dump({
        "classes": train_ds.classes,
        "prompts": [label_to_prompt(c) for c in train_ds.classes]
    }, f, indent=2)
print('Labels saved to clip_labels.json ✓')
print('\nFirst 5 class → prompt mappings:')
for c in train_ds.classes[:5]:
    print(f'  {c:45s} → "{label_to_prompt(c)}"')

## Section 8 — Optimizer, Loss & Text Embeddings

**Optimizer: AdamW**  
Only the unfrozen parameters are passed. AdamW adds decoupled weight decay (better than L2 regularization for transformers).

**Loss: CrossEntropyLoss**  
We treat this as a standard multi-class classification problem. The CLIP similarity scores act as logits.

**Text tokenization:**  
We tokenize all 38 class prompts once and move them to the GPU. During training, text features are re-encoded each epoch (because the text encoder is partially frozen, they stay stable, but re-encoding is cleaner).

**The logit scale = 100.0:**  
CLIP normally uses a learned temperature parameter. We fix it at 100 to produce sharp probability distributions — values below ~10 produce overly uniform distributions.

In [ ]:
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR,
    weight_decay=WEIGHT_DECAY
)
loss_fn = nn.CrossEntropyLoss()

# Tokenize all class text prompts once (reused every epoch)
all_prompts = [label_to_prompt(c) for c in train_ds.classes]
text_tokens = clip.tokenize(all_prompts, truncate=True).to(DEVICE)

print(f'Optimizer : AdamW | LR={LR} | WeightDecay={WEIGHT_DECAY}')
print(f'Loss fn   : CrossEntropyLoss')
print(f'Text tokens shape: {text_tokens.shape}  ({len(all_prompts)} prompts, max 77 tokens each)')

## Section 9 — Training Loop

The training loop implements:

1. **Text feature recomputation** at the start of each epoch (with `torch.no_grad()`)
2. **Forward pass**: encode image → normalize → compute cosine similarity with text features × 100
3. **NaN/Inf detection**: skip bad batches instead of crashing
4. **Gradient clipping**: prevents explosion (max norm = 1.0)
5. **Validation loop**: same forward pass, no gradient computation
6. **Best model saving**: saves whenever validation accuracy improves
7. **Early stopping**: stops if no improvement for 3 consecutive epochs

**Why cosine similarity × 100?**  
CLIP image/text features are L2-normalized, so their dot product = cosine similarity in [−1, 1]. Multiplying by 100 maps this to [−100, 100], giving CrossEntropyLoss meaningful gradients.

In [ ]:
history = {'train_acc': [], 'train_loss': [], 'val_acc': [], 'val_loss': [], 'nan_batches': []}
best_val_acc = 0.0
best_epoch   = 0
patience     = 3
patience_counter = 0

start_time = datetime.now()
print(f'Training started at {start_time.strftime("%H:%M:%S")}\n')

for epoch in range(EPOCHS):
    model.train()

    # Recompute text features (frozen, so they don't change — but good practice)
    with torch.no_grad():
        text_features = model.encode_text(text_tokens)
        text_features = nn.functional.normalize(text_features, dim=-1)

    total_loss = 0.0; correct = 0; total = 0; nan_count = 0

    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Train]')
    for images, labels in pbar:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        # Forward pass
        image_features = model.encode_image(images)
        image_features = nn.functional.normalize(image_features, dim=-1)
        logits = 100.0 * (image_features @ text_features.T)  # [batch, num_classes]
        loss   = loss_fn(logits, labels)

        # Skip bad batches
        if torch.isnan(loss) or torch.isinf(loss):
            nan_count += 1
            optimizer.zero_grad()
            continue

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP)
        optimizer.step()

        total_loss += loss.item()
        correct    += (logits.argmax(dim=-1) == labels).sum().item()
        total      += labels.size(0)
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{correct/total*100:.1f}%'})

    train_acc = correct / total * 100 if total > 0 else 0.0
    avg_loss  = total_loss / max(len(train_loader) - nan_count, 1)

    # ── Validation ────────────────────────────────────────────────────────
    model.eval()
    val_correct = 0; val_total = 0; val_loss_total = 0.0

    with torch.no_grad():
        text_features = model.encode_text(text_tokens)
        text_features = nn.functional.normalize(text_features, dim=-1)

        for images, labels in tqdm(val_loader, desc=f'Epoch {epoch+1}/{EPOCHS} [Val]  ', leave=False):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            image_features = model.encode_image(images)
            image_features = nn.functional.normalize(image_features, dim=-1)
            logits         = 100.0 * (image_features @ text_features.T)
            loss           = loss_fn(logits, labels)
            val_correct   += (logits.argmax(dim=-1) == labels).sum().item()
            val_total     += labels.size(0)
            val_loss_total += loss.item()

    val_acc  = val_correct / val_total * 100
    val_loss = val_loss_total / len(val_loader)

    history['train_acc'].append(train_acc)
    history['train_loss'].append(avg_loss)
    history['val_acc'].append(val_acc)
    history['val_loss'].append(val_loss)
    history['nan_batches'].append(nan_count)

    print(f'\nEpoch {epoch+1}/{EPOCHS}')
    print(f'  Train acc : {train_acc:.2f}%  |  Val acc : {val_acc:.2f}%')
    print(f'  Train loss: {avg_loss:.4f}   |  Val loss: {val_loss:.4f}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc; best_epoch = epoch + 1
        torch.save(model.state_dict(), 'clip_plant_best.pt')
        print(f'  ✓ NEW BEST — {val_acc:.2f}% (epoch {best_epoch})')
        patience_counter = 0
    else:
        patience_counter += 1
        print(f'  No improvement — patience {patience_counter}/{patience}')

    if patience_counter >= patience:
        print(f'\nEarly stopping after {patience} epochs without improvement.')
        break

duration = datetime.now() - start_time
print(f'\nTraining complete in {duration}')
print(f'Best val accuracy: {best_val_acc:.2f}% (epoch {best_epoch})')

## Section 10 — Visualize Results

Now we generate all three visualization plots:
1. Training curves (saved to `training_outputs/training_curves.png`)
2. Confusion matrix on best model (saved to `training_outputs/confusion_matrix.png`)
3. Per-class accuracy bar chart (saved to `training_outputs/per_class_accuracy.png`)

We reload the **best saved checkpoint** for final evaluation to ensure the most accurate confusion matrix.

In [ ]:
# 1. Training curves
curves_path = os.path.join(SAVE_DIR, 'training_curves.png')
plot_training_curves(history, curves_path)

In [ ]:
# 2. Reload best model → run full val evaluation → confusion matrix
print('Reloading best model checkpoint...')
model.load_state_dict(torch.load('clip_plant_best.pt', map_location=DEVICE))
model.eval()

all_preds_final = []; all_labels_final = []

with torch.no_grad():
    text_features = model.encode_text(text_tokens)
    text_features = nn.functional.normalize(text_features, dim=-1)

    for images, labels in tqdm(val_loader, desc='Final evaluation'):
        images = images.to(DEVICE)
        image_features = model.encode_image(images)
        image_features = nn.functional.normalize(image_features, dim=-1)
        logits = 100.0 * (image_features @ text_features.T)
        preds  = logits.argmax(dim=-1)
        all_preds_final.extend(preds.cpu().numpy())
        all_labels_final.extend(labels.numpy())

cm_path = os.path.join(SAVE_DIR, 'confusion_matrix.png')
cm, cm_norm = plot_confusion_matrix(all_labels_final, all_preds_final, train_ds.classes, cm_path)

In [ ]:
# 3. Per-class accuracy
per_class_path = os.path.join(SAVE_DIR, 'per_class_accuracy.png')
plot_per_class_accuracy(cm_norm, train_ds.classes, per_class_path)

## Section 11 — Save Training History

Save a complete JSON log of training metrics for later analysis or to reproduce plots without re-running training.

In [ ]:
end_time = datetime.now()
duration = end_time - start_time

history_path = os.path.join(SAVE_DIR, 'training_history.json')
with open(history_path, 'w') as f:
    json.dump({
        'train_acc':        history['train_acc'],
        'val_acc':          history['val_acc'],
        'train_loss':       history['train_loss'],
        'val_loss':         history['val_loss'],
        'best_val_acc':     best_val_acc,
        'best_epoch':       best_epoch,
        'duration_seconds': duration.total_seconds(),
        'config': {'epochs': EPOCHS, 'batch_size': BATCH_SIZE, 'lr': LR,
                   'weight_decay': WEIGHT_DECAY, 'grad_clip': GRAD_CLIP}
    }, f, indent=2)

print('=' * 60)
print('TRAINING COMPLETE')
print('=' * 60)
print(f'Best val accuracy  : {best_val_acc:.2f}% (epoch {best_epoch})')
print(f'Final train acc    : {history["train_acc"][-1]:.2f}%')
print(f'Total training time: {duration}')
print(f'Model checkpoint   : clip_plant_best.pt')
print(f'Labels file        : clip_labels.json')
print(f'History saved      : {history_path}')
print('=' * 60)